In [6]:
import tensorflow as tf
from keras.models import load_model, Model  # Needed if the architecture is redefined.
from keras.layers import GlobalAveragePooling2D, Dense, Dropout  # Needed if the architecture is redefined.
from keras.utils import load_img, img_to_array
from keras.applications.mobilenet_v2 import preprocess_input  # MobileNetV2 preprocessing function.
import numpy as np
import os
import json  # Read the class-index mapping.
from pathlib import Path

# Find the project root when the notebook is run from the project root or notebooks/.
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "dogs.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root containing data/dogs.csv.")

APP_DIR = PROJECT_ROOT / "app"

# --- Configuration ---
IMG_SIZE = 128  # Must match the image size used during training.
NUM_CLASSES = 70  # Number of output classes in the saved model.

# Path to the saved model.
model_path = APP_DIR / "best_dog_breed_model.keras"

# --- Load class names ---
class_names_mapping = None
try:
    # Load class names from the JSON file.
    with open(APP_DIR / "class_indices.json", "r", encoding="utf-8") as f:
        class_indices = json.load(f)
        # Reverse the mapping so each index maps to a class name.
        class_names_mapping = {v: k for k, v in class_indices.items()} # JSON object keys are strings; the values are integer indices.
    print(f"Class names loaded from {APP_DIR / 'class_indices.json'}")
except FileNotFoundError:
    print("Error: class_indices.json not found.")
    # Define the class names manually here if the JSON file is unavailable.
    # Example:
    # class_names_mapping = {0: 'sharpei', 1: 'border_collie', ... } # List all 70 class names here.
    # If you define class_names_mapping manually, uncomment and edit this line.
    # class_names_mapping = {i: f'breed_{i}' for i in range(NUM_CLASSES)} # Example: breed_0, breed_1, ...
    print("Please provide class names manually or ensure app/class_indices.json exists.")
    # exit() # Exit if class labels are unavailable.

# --- Load the model ---
try:
    model = load_model(str(model_path))
    print(f"Model loaded successfully from {model_path}")

    # If load_model fails, redefine the architecture and load the weights instead.
    # (Use the same architecture as the training notebook.)
    # base_model = MobileNetV2(weights=None, include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    # model = Model(inputs=base_model.input, outputs=Dense(NUM_CLASSES, activation='softmax')(GlobalAveragePooling2D()(Dropout(0.5)(Dense(256, activation='relu')(base_model.output))))))
    # model.load_weights(model_path) # Set the weights file path here.
    # print("Model architecture redefined and weights loaded successfully.")

except Exception as e:
    print(f"Error loading model: {e}")
    exit()

# --- Image preprocessing function ---
def preprocess_image(img_path, target_size):
    try:
        img = load_img(str(img_path), target_size=(target_size, target_size))
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        return preprocess_input(img_array)
    except FileNotFoundError:
        print(f"Error: Image file not found at {img_path}")
        return None
    except Exception as e:
        print(f"Error processing image {img_path}: {e}")
        return None

# --- Prediction function for one image ---
def predict_dog_breed(image_path, model, class_map, img_size):
    processed_image = preprocess_image(image_path, img_size)

    if processed_image is not None:
        try:
            predictions = model.predict(processed_image)

            if class_map:
                predicted_class_index = np.argmax(predictions[0])
                predicted_breed = class_map.get(predicted_class_index, "Unknown Breed")
                confidence = np.max(predictions[0]) * 100

                print(f"\nPrediction for: {os.path.basename(image_path)}")
                print(f"  Breed: {predicted_breed}")
                print(f"  Confidence: {confidence:.2f}%")

                # Display the five most likely predictions.
                top_5_indices = np.argsort(predictions[0])[-5:][::-1]
                print("\nTop 5 predictions:")
                for i in top_5_indices:
                    breed = class_map.get(i, "Unknown Breed")
                    prob = predictions[0][i] * 100
                    print(f"  - {breed}: {prob:.2f}%")
            else:
                print("\nPrediction probabilities (no class names available):")
                print(predictions[0])

        except Exception as e:
            print(f"Error during prediction: {e}")
    else:
        print("Image preprocessing failed.")

# --- Example prediction ---
if __name__ == "__main__":
    # Path to the image to classify.
    new_image_to_predict = PROJECT_ROOT / "image" / "german_baby.jpg"

    if class_names_mapping:  # Predict only when class names are available.
        predict_dog_breed(new_image_to_predict, model, class_names_mapping, IMG_SIZE)
    else:
        print("Cannot perform prediction because class names are not available.")


Class names loaded from class_indices.json
Model loaded successfully from best_dog_breed_model.keras
1/1 [==============================] - 1s 1s/step

Prediction for: german_baby.jpg
  Breed: Airedale
  Confidence: 44.34%

Top 5 predictions:
  - Airedale: 44.34%
  - German Sheperd: 44.04%
  - Irish Wolfhound: 4.03%
  - Yorkie: 3.35%
  - Malinois: 2.10%
